# Evaluate trained DeepLOB models

This notebook loads the three saved checkpoints from `Output_models` and evaluates each one on the same test split logic used in the training notebooks.

In [9]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from deeplob_eval import DeepLOB5Stable, run_checkpoint_evaluation

In [10]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

2.7.0+cu128
CUDA available: True
CUDA device count: 1
GPU: NVIDIA GeForce RTX 5090
Using device: cuda


In [17]:
PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / "Output_models"

experiments = [
    {
        "name": "k10",
        "data_path": PROJECT_ROOT / "test_10.parquet",
        "checkpoint_path": MODEL_DIR / "deeplob5_stabilized_best_10.pth",
        "has_index_column": False,
    },
    {
        "name": "k_20",
        "data_path": PROJECT_ROOT / "test_Xy.parquet",
        "checkpoint_path": MODEL_DIR / "deeplob5_stabilized_best.pth",
        "has_index_column": True,
    },
    {
        "name": "k50",
        "data_path": PROJECT_ROOT / "test_50.parquet",
        "checkpoint_path": MODEL_DIR / "deeplob5_stabilized_best_50.pth",
        "has_index_column": False,
    },
    
]

for exp in experiments:
    assert exp["data_path"].exists(), exp["data_path"]
    assert exp["checkpoint_path"].exists(), exp["checkpoint_path"]

pd.DataFrame(
    {
        "name": exp["name"],
        "data": exp["data_path"].name,
        "checkpoint": exp["checkpoint_path"].relative_to(PROJECT_ROOT).as_posix(),
        "has_index_column": exp["has_index_column"],
    }
    for exp in experiments
)

,name,data,checkpoint,has_index_column
0,k10,test_10.parquet,Output_models/deeplob5_stabilized_best_10.pth,False
1,k_20,test_Xy.parquet,Output_models/deeplob5_stabilized_best.pth,True
2,k50,test_50.parquet,Output_models/deeplob5_stabilized_best_50.pth,False


In [18]:
results = {}
summary_rows = []


def evaluate_and_display(exp):
    print("Data:", exp["data_path"].name)
    print("Checkpoint:", exp["checkpoint_path"].relative_to(PROJECT_ROOT).as_posix())
    print()

    result = run_checkpoint_evaluation(
        data_path=exp["data_path"],
        checkpoint_path=exp["checkpoint_path"],
        has_index_column=exp["has_index_column"],
        device=device,
    )
    results[exp["name"]] = result

    checkpoint = result["checkpoint"]
    prepared = result["prepared"]
    test_metrics = result["test_metrics"]
    per_class_df = result["per_class_metrics"]

    print("Loaded best_epoch:", checkpoint.get("best_epoch"))
    print("Loaded best_val_f1:", checkpoint.get("best_val_f1"))
    print()
    print("Shape:", prepared.df.shape)
    print("Columns:", prepared.df.columns.tolist())
    print()
    print("Index head/tail:", prepared.df["index"].iloc[0], prepared.df["index"].iloc[-1])
    print()
    print("Raw label distribution:")
    print(prepared.df["label"].value_counts(dropna=False).sort_index())
    print()
    print("Raw label share:")
    print(prepared.df["label"].value_counts(dropna=False, normalize=True).sort_index())
    print()
    print("Split sizes:")
    print("train:", prepared.split_sizes["train"])
    print("valid:", prepared.split_sizes["valid"])
    print("test :", prepared.split_sizes["test"])
    print("test windows:", prepared.split_sizes["test_windows"])
    print()
    print("Effective train class counts:", prepared.class_counts)
    print("Class weights:", prepared.class_weights)
    print()
    print("Best model test metrics:")
    print(test_metrics)
    print()
    print("Confusion Matrix:")
    display(result["confusion_matrix"])
    print()
    print("Per-class metrics:")
    display(per_class_df)
    print()
    print("Macro F1 from per-class table:", per_class_df["f1"].mean())
    print()
    return result


def add_summary_row(exp, result):
    checkpoint = result["checkpoint"]
    test_metrics = result["test_metrics"]
    summary_rows.append(
        {
            "name": exp["name"],
            "data": exp["data_path"].name,
            "checkpoint": exp["checkpoint_path"].name,
            "best_epoch": checkpoint.get("best_epoch"),
            "best_val_f1": checkpoint.get("best_val_f1"),
            "test_loss": test_metrics["loss"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
        }
    )

## k10 model on test_10

In [19]:
exp = experiments[0]
result = evaluate_and_display(exp)
results[exp["name"]] = result
add_summary_row(exp, result)

Data: test_10.parquet
Checkpoint: Output_models/deeplob5_stabilized_best_10.pth

Loaded best_epoch: 27
Loaded best_val_f1: 0.569407166602569

Shape: (8774466, 23)
Columns: ['index', 'BidPrice5', 'BidVolume5', 'BidPrice4', 'BidVolume4', 'BidPrice3', 'BidVolume3', 'BidPrice2', 'BidVolume2', 'BidPrice1', 'BidVolume1', 'AskPrice5', 'AskVolume5', 'AskPrice4', 'AskVolume4', 'AskPrice3', 'AskVolume3', 'AskPrice2', 'AskVolume2', 'AskPrice1', 'AskVolume1', 'label', 'label_cls']

Index head/tail: 0 8774465

Raw label distribution:
label
-1.0    1078043
 0.0    6614408
 1.0    1082015
Name: count, dtype: int64

Raw label share:
label
-1.0    0.122861
 0.0    0.753825
 1.0    0.123314
Name: proportion, dtype: float64

Split sizes:
train: (6142126, 23)
valid: (1316170, 23)
test : (1316170, 23)
test windows: 1316071

Effective train class counts: [ 759077 4623680  759270]
Class weights: tensor([2.6971, 0.4428, 2.6965], device='cuda:0')

Best model test metrics:
{'loss': 0.7700965669239123, 'accuracy

,pred_down,pred_stationary,pred_up
true_down,99782,57872,4166
true_stationary,150324,714794,127034
true_up,5329,62824,93946



Per-class metrics:


,class,precision,recall,f1,support
0,down,0.390636,0.616623,0.478278,161820
1,stationary,0.855539,0.720448,0.782204,992152
2,up,0.417267,0.579559,0.485202,162099



Macro F1 from per-class table: 0.581894554247735



## k20 model on test_Xy

In [20]:
exp = experiments[2]
result = evaluate_and_display(exp)
results[exp["name"]] = result
add_summary_row(exp, result)

Data: test_50.parquet
Checkpoint: Output_models/deeplob5_stabilized_best_50.pth

Loaded best_epoch: 11
Loaded best_val_f1: 0.46347800351496193

Shape: (8759986, 23)
Columns: ['index', 'BidPrice5', 'BidVolume5', 'BidPrice4', 'BidVolume4', 'BidPrice3', 'BidVolume3', 'BidPrice2', 'BidVolume2', 'BidPrice1', 'BidVolume1', 'AskPrice5', 'AskVolume5', 'AskPrice4', 'AskVolume4', 'AskPrice3', 'AskVolume3', 'AskPrice2', 'AskVolume2', 'AskPrice1', 'AskVolume1', 'label', 'label_cls']

Index head/tail: 0 8759985

Raw label distribution:
label
-1.0    1162732
 0.0    6440186
 1.0    1157068
Name: count, dtype: int64

Raw label share:
label
-1.0    0.132732
 0.0    0.735182
 1.0    0.132086
Name: proportion, dtype: float64

Split sizes:
train: (6131990, 23)
valid: (1313998, 23)
test : (1313998, 23)
test windows: 1313899

Effective train class counts: [ 812541 4512703  806647]
Class weights: tensor([2.5155, 0.4529, 2.5339], device='cuda:0')

Best model test metrics:
{'loss': 0.9834550531277062, 'accura

,pred_down,pred_stationary,pred_up
true_down,62543,89319,26117
true_stationary,112198,653935,195736
true_up,14137,76926,82988



Per-class metrics:


,class,precision,recall,f1,support
0,down,0.331129,0.351407,0.340967,177979
1,stationary,0.797307,0.679859,0.733914,961869
2,up,0.272234,0.476803,0.346583,174051



Macro F1 from per-class table: 0.47382120288099466



## k50 model on test_50

In [21]:
exp = experiments[1]
result = evaluate_and_display(exp)
results[exp["name"]] = result
add_summary_row(exp, result)

Data: test_Xy.parquet
Checkpoint: Output_models/deeplob5_stabilized_best.pth

Loaded best_epoch: 42
Loaded best_val_f1: 0.5852603995943448

Shape: (8770846, 23)
Columns: ['index', 'BidPrice5', 'BidVolume5', 'BidPrice4', 'BidVolume4', 'BidPrice3', 'BidVolume3', 'BidPrice2', 'BidVolume2', 'BidPrice1', 'BidVolume1', 'AskPrice5', 'AskVolume5', 'AskPrice4', 'AskVolume4', 'AskPrice3', 'AskVolume3', 'AskPrice2', 'AskVolume2', 'AskPrice1', 'AskVolume1', 'label', 'label_cls']

Index head/tail: 0 8770845

Raw label distribution:
label
-1.0    1116268
 0.0    6538661
 1.0    1115917
Name: count, dtype: int64

Raw label share:
label
-1.0    0.127270
 0.0    0.745499
 1.0    0.127230
Name: proportion, dtype: float64

Split sizes:
train: (6139592, 23)
valid: (1315627, 23)
test : (1315627, 23)
test windows: 1315528

Effective train class counts: [ 781983 4578358  779152]
Class weights: tensor([2.6171, 0.4470, 2.6266], device='cuda:0')

Best model test metrics:
{'loss': 0.7621799761130518, 'accuracy':

,pred_down,pred_stationary,pred_up
true_down,98626,68783,3472
true_stationary,121617,725357,128113
true_up,3369,66295,99896



Per-class metrics:


,class,precision,recall,f1,support
0,down,0.441059,0.577162,0.500014,170881
1,stationary,0.843012,0.743890,0.790355,975087
2,up,0.431552,0.589148,0.498183,169560



Macro F1 from per-class table: 0.5961841415744066



In [22]:
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,name,data,checkpoint,best_epoch,best_val_f1,test_loss,test_accuracy,test_macro_f1
0,k10,test_10.parquet,deeplob5_stabilized_best_10.pth,27,0.569407,0.770097,0.690329,0.581895
1,k50,test_50.parquet,deeplob5_stabilized_best_50.pth,11,0.463478,0.983455,0.608468,0.473821
2,k_20,test_Xy.parquet,deeplob5_stabilized_best.pth,42,0.585260,0.762180,0.702288,0.596184
